# Engenharia de Freatures: The Movies Dataset
Esta etapa é a mais complexa do pipeline de dados de filmes. Ela transforma colunas armazenadas em formato JSON serializado em estruturas Python nativas, aplica uma **regra de negócio baseada no Princípio de Pareto** para correção de franquias ausentes, e cria features derivadas que habilitam análises comparativas entre filmes independentes e propriedades intelectuais de franquia.

- **Escopo:** Deserialização de colunas JSON, correção de franquias por mapeamento explícito e criação de features binárias e de granularidade
- **Produto desta etapa:** `movies_feature_engineered.csv` — dataset analiticamente completo para os notebooks de análise

In [1]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..')))


# Importação de Bibliotecas e Módulos
import ast
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Nossos módulos customizados da pasta src/
from src import load_data_csv

---
## Carregando Dados

In [2]:
caminho = '../data/processed/movies_cleaned.csv'
df_movies = load_data_csv(caminho)

Dados carregados com sucesso! Formato: (41184, 18)


---
## Correção de Franquias: Aplicação da Regra de Pareto ao Enriquecimento de Dados

A coluna `belongs_to_collection` do dataset original apresenta ausências para filmes de grandes franquias — não por ausência de vínculo, mas por limitações de cobertura da base de dados na época de sua criação. Como o pertencimento a uma franquia é uma das dimensões analíticas mais relevantes deste projeto, a ausência desses vínculos introduziria viés sistemático nas comparações entre filmes independentes e de franquia.

A correção prioriza as **franquias de maior impacto comercial e cultural**, seguindo o princípio de que um subconjunto reduzido de propriedades intelectuais representa a maior parte da receita de bilheteria global. O mapeamento explícito por palavra-chave no título garante que os filmes mais analiticamente relevantes estejam corretamente classificados.

- **Regra de negócio:** Somente registros com `belongs_to_collection` ausente recebem a correção — vínculos existentes no dataset original nunca são sobrescritos
- **Escopo da correção:** Franquias do universo Marvel, DC, grandes sagas de animação, ação e aventura que compõem o topo da distribuição de bilheteria histórica

In [3]:
# Dicionário Expandido: Mapeamento das Maiores Franquias do Cinema 
correcoes_franquias = {
    # 🦸‍♂️ Heróis (Marvel & DC)
    'Avengers': 'The Avengers Collection',
    'Vingadores': 'The Avengers Collection',
    'Spider-Man': 'Spider-Man Collection',
    'Homem-Aranha': 'Spider-Man Collection',
    'Iron Man': 'Iron Man Collection',
    'Homem de Ferro': 'Iron Man Collection',
    'Captain America': 'Captain America Collection',
    'Thor': 'Thor Collection',
    'X-Men': 'X-Men Collection',
    'Batman': 'Batman Collection',
    'Dark Knight': 'Batman Collection',
    'Cavaleiro das Trevas': 'Batman Collection',
    'Superman': 'Superman Collection',
    
    # 🧙‍♂️ Sci-Fi & Fantasia
    'Star Wars': 'Star Wars Collection',
    'Harry Potter': 'Harry Potter Collection',
    'Lord of the Rings': 'The Lord of the Rings Collection',
    'Senhor dos Anéis': 'The Lord of the Rings Collection',
    'Hobbit': 'The Hobbit Collection',
    'Matrix': 'The Matrix Collection',
    'Hunger Games': 'The Hunger Games Collection',
    'Jogos Vorazes': 'The Hunger Games Collection',
    'Twilight': 'The Twilight Saga Collection',
    'Crepúsculo': 'The Twilight Saga Collection',
    'Jumanji': 'Jumanji Collection',

    # 💥 Ação, Aventura & Sci-Fi
    'James Bond': 'James Bond Collection',
    '007': 'James Bond Collection',
    'Fast and Furious': 'The Fast and the Furious Collection',
    'Velozes e Furiosos': 'The Fast and the Furious Collection',
    'Mission: Impossible': 'Mission: Impossible Collection',
    'Missão: Impossível': 'Mission: Impossible Collection',
    'Indiana Jones': 'Indiana Jones Collection',
    'Pirates of the Caribbean': 'Pirates of the Caribbean Collection',
    'Piratas do Caribe': 'Pirates of the Caribbean Collection',
    'Jurassic': 'Jurassic Park Collection', 
    'Terminator': 'The Terminator Collection',
    'Exterminador do Futuro': 'The Terminator Collection',
    'Transformers': 'Transformers Collection',
    'Mad Max': 'Mad Max Collection',

    # 🎬 Animações Gigantes
    'Toy Story': 'Toy Story Collection',
    'Shrek': 'Shrek Collection',
    'Despicable Me': 'Despicable Me Collection',
    'Meu Malvado Favorito': 'Despicable Me Collection',
    'Minions': 'Despicable Me Collection',
    'Ice Age': 'Ice Age Collection',
    'A Era do Gelo': 'Ice Age Collection'
}

In [4]:
# O Loop Inteligente: Varrer o dicionário e aplicar a regra
for palavra_chave, nome_franquia in correcoes_franquias.items():
    filtro = df_movies['title'].str.contains(palavra_chave, case=False, na=False)
    df_movies.loc[filtro & df_movies['belongs_to_collection'].isna(), 'belongs_to_collection'] = nome_franquia
    df_movies.loc[filtro, 'is_franchise'] = True

print(f"Correções manuais aplicadas para {len(correcoes_franquias)} chaves de franquias!")

Correções manuais aplicadas para 45 chaves de franquias!


---
## Deserialização de Colunas Estruturadas: De JSON Serializado para Listas Nativas

Colunas como `genres`, `production_countries`, `spoken_languages` e `belongs_to_collection` chegam do CSV como strings que representam estruturas JSON — um formato que preserva a estrutura original, mas torna os dados **opacos para qualquer operação analítica**. Sem a extração dos valores internos, é impossível filtrar por gênero, agrupar por país de produção ou identificar pertencimento a franquias.

- **Decisão arquitetural:** A extração é centralizada em funções reutilizáveis para garantir tratamento uniforme e seguro de nulos em todas as colunas afetadas
- **Resultado:** Cada coluna passa a armazenar uma lista Python nativa, viabilizando operações de explode, filtragem e contagem nas análises subsequentes

In [5]:
# Função para extrair os nomes dos diretores, atores e gêneros a partir do formato JSON/Dicionário

def extrair_nomes(texto):
    # Se o valor for nulo, retorna uma lista vazia
    if pd.isna(texto):
        return []
    
    try:
        # ast.literal_eval converte a string "[]" em uma lista real do Python
        lista_dicionarios = ast.literal_eval(texto)
        
        # Extrai apenas o 'name' de cada dicionário dentro da lista
        return [item['name'] for item in lista_dicionarios]
    
    except (ValueError, SyntaxError):
        # Se der erro na conversão de algum registro corrompido, ignora e retorna vazio
        return []

In [6]:
# Aplicando a função nas colunas complexas
colunas_para_extrair = ['genres', 'production_countries', 'spoken_languages']

for coluna in colunas_para_extrair:
    # O .apply joga a nossa função linha por linha na coluna
    df_movies[coluna] = df_movies[coluna].apply(extrair_nomes)

# Visualizando o resultado limpo
display(df_movies[['title', 'genres', 'production_countries', 'spoken_languages']].head())

,title,genres,production_countries,spoken_languages
0,Toy Story,"[Animation, Comedy, Family]",[United States of America],[English]
1,Jumanji,"[Adventure, Fantasy, Family]",[United States of America],"[English, Français]"
2,Grumpier Old Men,"[Romance, Comedy]",[United States of America],[English]
3,Waiting to Exhale,"[Comedy, Drama, Romance]",[United States of America],[English]
4,Father of the Bride Part II,[Comedy],[United States of America],[English]


In [7]:
# Função para extrair a coleção (franquia) a partir do formato JSON/Dicionário

def extrair_colecao(texto):
    if pd.isna(texto):
        return np.nan
    try:
        dicionario = ast.literal_eval(texto)
        return dicionario['name'] # Retorna direto a string, não uma lista
    except:
        return np.nan

In [8]:
df_movies['belongs_to_collection'] = df_movies['belongs_to_collection'].apply(extrair_colecao)

In [9]:
# Visualizando o resultado limpo
display(df_movies[['title','belongs_to_collection', 'genres', 'production_countries', 'spoken_languages']].head())

,title,belongs_to_collection,genres,production_countries,spoken_languages
0,Toy Story,Toy Story Collection,"[Animation, Comedy, Family]",[United States of America],[English]
1,Jumanji,NaN,"[Adventure, Fantasy, Family]",[United States of America],"[English, Français]"
2,Grumpier Old Men,Grumpy Old Men Collection,"[Romance, Comedy]",[United States of America],[English]
3,Waiting to Exhale,NaN,"[Comedy, Drama, Romance]",[United States of America],[English]
4,Father of the Bride Part II,Father of the Bride Collection,[Comedy],[United States of America],[English]


---
## Criação de Feature Derivada: Flag de Franquia (`is_franchise`)

Uma das hipóteses centrais deste projeto é que filmes pertencentes a franquias consolidadas apresentam padrões de desempenho financeiro e de recepção distintos dos filmes independentes. Para operacionalizar essa hipótese nas análises, é necessário uma feature binária que permita segmentar e comparar os dois grupos de forma direta.

- **Regra de negócio:** Um filme é classificado como `is_franchise = True` se e somente se possui um vínculo de coleção identificado — seja pelo dado original, seja pela correção por mapeamento de franquias aplicada na etapa anterior
- **Valor analítico:** Esta feature atua como variável de segmentação primária em todas as análises comparativas de ROI, bilheteria e avaliação do público

In [10]:
# Criando uma coluna is_franchise para indicar se o filme pertence a uma franquia ou não
df_movies['is_franchise'] = df_movies['belongs_to_collection'].notna().astype(bool)
# Visualizando o resultado final
display(df_movies[['title','belongs_to_collection', 'is_franchise', 'genres', 'production_countries', 'spoken_languages']].head())

,title,belongs_to_collection,is_franchise,genres,production_countries,spoken_languages
0,Toy Story,Toy Story Collection,True,"[Animation, Comedy, Family]",[United States of America],[English]
1,Jumanji,NaN,False,"[Adventure, Fantasy, Family]",[United States of America],"[English, Français]"
2,Grumpier Old Men,Grumpy Old Men Collection,True,"[Romance, Comedy]",[United States of America],[English]
3,Waiting to Exhale,NaN,False,"[Comedy, Drama, Romance]",[United States of America],[English]
4,Father of the Bride Part II,Father of the Bride Collection,True,[Comedy],[United States of America],[English]


---
## Checkpoint de Persistência: Separação entre Enriquecimento e Desnormalização

O dataset é salvo neste ponto — após o enriquecimento de features e antes de qualquer operação de explosão de colunas — para preservar a estrutura **um-filme-por-linha**. Essa separação é uma decisão arquitetural deliberada: análises financeiras e de atributos agregados do filme (orçamento, receita, `is_franchise`) consomem este artefato; análises por gênero ou país consumirão versões explodidas derivadas dele, mantendo a rastreabilidade entre as camadas do pipeline.

In [11]:
# Salvando o DataFrame processado para a próxima etapa de análise
caminho_saida = '../data/processed/movies_enriched.csv'
df_movies.to_csv(caminho_saida, index=False)

---
##  Modelagem Dimensional: Atomização de Listas Complexas

O dataset enriquecido armazena informações cruciais (gêneros, países de produção e idiomas) no formato de listas dentro de uma única célula. Embora eficiente para armazenamento, essa estrutura **inviabiliza análises categóricas isoladas**. Para responder a perguntas de negócio como "qual país produz os filmes mais rentáveis?" ou "qual o ROI médio do gênero terror?", é imperativo normalizar essas variáveis para um modelo *one-to-many*.

- **Decisão arquitetural (Prevenção de Produto Cartesiano):** Optou-se por **não** explodir múltiplas colunas no mesmo *dataframe*. Fazer isso geraria uma multiplicação exponencial das linhas (ex: Gênero $\times$ País $\times$ Idioma), o que corromperia gravemente o cálculo de métricas financeiras como `revenue` e `budget`. Em vez disso, adotamos a criação de **tabelas dimensão independentes** para cada categoria.
- **Trade-off aceito:** O projeto passará a gerenciar múltiplos arquivos `.csv` (uma base principal estática e bases dimensionais explodidas) em detrimento de uma tabela única universal. Essa fragmentação intencional blinda a integridade estatística dos dados globais enquanto habilita mergulhos analíticos profundos em categorias específicas.
---

### 1. Dimensão de Gêneros (`movies_genres`)

Isolamento da variável `genres` para viabilizar análises de volume e performance financeira por categoria cinematográfica. 

* **Operação:** Fatiamento das listas e aplicação do `.explode()`.
* **Validação:** Inspeção da nova granularidade (múltiplas linhas por filme) e integridade dos tipos.
* **Persistência:** Exportação da tabela dimensão para `data/processed/movies_genres.csv`.

In [12]:
# Explodindo a coluna de gêneros para criar uma linha por gênero
df_genres_exploded = df_movies.explode('genres')

# Limpando os espaços em branco dos gêneros
df_genres_exploded['genres'] = df_genres_exploded['genres'].str.strip()

# Resetando o índice para evitar índices duplicados
df_genres_exploded.reset_index(drop=True, inplace=True)

In [13]:
# Visualizando o resultado
display(
    df_genres_exploded[['title', 'genres']]
    .head(20)
    .style

    .set_properties(
        subset=['title'],
        **{
            'background-color': '#D8E2DC',
            'color': 'black'
        }
    )

    .set_properties(
        subset=['genres'],
        **{
            'background-color': "#6D597A",
            'color': 'white'
        }
    )
)     

,title,genres
0,Toy Story,Animation
1,Toy Story,Comedy
2,Toy Story,Family
3,Jumanji,Adventure
4,Jumanji,Fantasy
5,Jumanji,Family
6,Grumpier Old Men,Romance
7,Grumpier Old Men,Comedy
8,Waiting to Exhale,Comedy
9,Waiting to Exhale,Drama


In [14]:
# Verificando o número de linhas após a explosão
display(f"Número de linhas após a explosão: {df_genres_exploded.shape[0]}")

'Número de linhas após a explosão: 87099'

In [15]:
# Salvando o DataFrame explodido para análise de gêneros
caminho_saida_genres_exploded = '../data/processed/movies_genres.csv'
df_genres_exploded.to_csv(caminho_saida_genres_exploded, index=False)

### 2. Dimensão de Países de Produção (`movies_countries`)

Extração da variável `production_countries` para mapeamento geográfico da indústria e análise comparativa de orçamentos regionais (ex: Hollywood vs. Bollywood).

* **Operação:** Fatiamento das listas e aplicação do `.explode()`.
* **Validação:** Verificação de nulos resultantes e expansão do número de linhas.
* **Persistência:** Exportação da tabela dimensão para `data/processed/movies_countries.csv`.

In [16]:
# Explodindo a coluna de países para criar uma linha por país
df_countries_exploded = df_movies.explode('production_countries')

# Limpando os espaços em branco dos países
df_countries_exploded['production_countries'] = df_countries_exploded['production_countries'].str.strip()

# Resetando o índice para evitar índices duplicados
df_countries_exploded.reset_index(drop=True, inplace=True)

In [17]:
# Visualizando o resultado
display(
    df_countries_exploded[['title', 'production_countries']]
    .head(20)
    .style

    .set_properties(
        subset=['title'],
        **{
            'background-color': '#D8E2DC',
            'color': 'black'
        }
    )

    .set_properties(
        subset=['production_countries'],
        **{
            'background-color': "#588157",
            'color': 'white'
        }
    )
)     

,title,production_countries
0,Toy Story,United States of America
1,Jumanji,United States of America
2,Grumpier Old Men,United States of America
3,Waiting to Exhale,United States of America
4,Father of the Bride Part II,United States of America
5,Heat,United States of America
6,Sabrina,Germany
7,Sabrina,United States of America
8,Tom and Huck,United States of America
9,Sudden Death,United States of America


In [18]:
# Verificando o número de linhas após a explosão
display(f"Número de linhas após a explosão: {df_countries_exploded.shape[0]}")

'Número de linhas após a explosão: 50978'

In [19]:
# Salvando o DataFrame explodido para análise de países
caminho_saida_countries_exploded = '../data/processed/movies_countries.csv'
df_countries_exploded.to_csv(caminho_saida_countries_exploded, index=False)

### 3. Dimensão de Idiomas Falados (`movies_languages`)

Separação da variável `spoken_languages` para futuras análises de alcance global e correlação entre idioma original e popularidade internacional.

* **Operação:** Fatiamento das listas e aplicação do `.explode()`.
* **Validação:** Checagem da estrutura atômica dos idiomas.
* **Persistência:** Exportação da tabela dimensão para `data/processed/movies_languages.csv`.

In [20]:
# Explodindo a coluna de idiomas para criar uma linha por idioma
df_languages_exploded = df_movies.explode('spoken_languages')

# Limpando os espaços em branco dos idiomas
df_languages_exploded['spoken_languages'] = df_languages_exploded['spoken_languages'].str.strip()

# Resetando o índice para evitar índices duplicados
df_languages_exploded.reset_index(drop=True, inplace=True)

In [21]:
# Visualizando o resultado
display(
    df_languages_exploded[['title', 'spoken_languages']]
    .head(20)
    .style

    .set_properties(
        subset=['title'],
        **{
            'background-color': '#D8E2DC',
            'color': 'black'
        }
    )

    .set_properties(
        subset=['spoken_languages'],
        **{
            'background-color': "#3D5A80",
            'color': 'white'
        }
    )
)     

,title,spoken_languages
0,Toy Story,English
1,Jumanji,English
2,Jumanji,Français
3,Grumpier Old Men,English
4,Waiting to Exhale,English
5,Father of the Bride Part II,English
6,Heat,English
7,Heat,Español
8,Sabrina,Français
9,Sabrina,English


In [22]:
# Verificando o número de linhas após a explosão
display(f"Número de linhas após a explosão: {df_languages_exploded.shape[0]}")

'Número de linhas após a explosão: 52455'

In [23]:
# Salvando o DataFrame explodido para análise de idiomas
caminho_saida_languages_exploded = '../data/processed/movies_languages.csv'
df_languages_exploded.to_csv(caminho_saida_languages_exploded, index=False)